In [0]:
drivers = [
    ("VER", "Max Verstappen", "Red Bull"),
    ("NOR", "Lando Norris", "McLaren"),
    ("LEC", "Charles Leclerc", "Ferrari")
]

columns = ["driver_code", "driver_name", "team"]

drivers_df = spark.createDataFrame(drivers, columns)

display(drivers_df)

In [0]:
from pyspark.sql import functions as F

# Keep only the columns needed for this view.
driver_teams_df = drivers_df.select("driver_name", "team")

display(driver_teams_df)

In [0]:
# Keep only Red Bull drivers.
red_bull_df = drivers_df.filter(
    F.col("team") == "Red Bull"
)

display(red_bull_df)

In [0]:
team_counts_df = (
    drivers_df
    .groupBy("team")
    .agg(
        F.count("*").alias("driver_count")
    )
    .orderBy("team")
)

display(team_counts_df)

In [0]:
drivers_with_label_df = drivers_df.withColumn(
    "driver_label",
    F.concat(
        F.col("driver_code"),
        F.lit(" - "),
        F.col("driver_name")
    )
)

display(drivers_with_label_df)

In [0]:
# Find the catalog available in your Databricks workspace.
catalog = spark.sql("SELECT current_catalog() AS catalog").first()["catalog"]

# Create a schema just for learning tables.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.f1_learning")

table_name = f"{catalog}.f1_learning.drivers_demo"

# Save the DataFrame as a managed Delta table.
(
    drivers_with_label_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)

print(f"Saved Delta table: {table_name}")

In [0]:
saved_drivers_df = spark.table(table_name)

display(saved_drivers_df)

In [0]:
display(spark.sql(f"DESCRIBE DETAIL {table_name}"))